# 02 — Baseline Model — Trace the Ace

Notebook B of the 4-notebook pipeline (see `Context.md`). Consumes
`trace-the-ace-features` (Notebook A's output) and produces the
`trace-the-ace-baseline` Kaggle Dataset consumed by Notebook D:

- `oof_predictions_baseline.csv` — out-of-fold predictions, columns
  `response_id, pred_baseline`
- `baseline_model.pkl` — final pipeline fit on all training data
- `baseline_metrics.json` — per-fold + overall log loss / AUC, base-rate floor

Sklearn logistic regression on engineered features only, matching the
reference DrivenData blog approach but using the full `feat_*` set from
Notebook A instead of just 3 hand-built features. CPU-only, no GPU needed --
this notebook's job is to sanity-check the CV harness before the expensive
LLM stage.


## 1. Setup

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score

warnings.filterwarnings("ignore")

def _resolve_dataset_dir(slug: str) -> Path:
    """Kaggle mounts a dataset at /kaggle/input/<slug>/ normally, but some
    environments nest it under /kaggle/input/datasets/<owner>/<slug>/ instead
    (observed in Notebook A) -- auto-detect whichever is present."""
    direct = Path("/kaggle/input") / slug
    if (direct / "train_features_enriched.csv").exists():
        return direct
    nested_root = Path("/kaggle/input/datasets")
    if nested_root.exists():
        for owner_dir in nested_root.iterdir():
            candidate = owner_dir / slug
            if (candidate / "train_features_enriched.csv").exists():
                return candidate
    raise FileNotFoundError(
        f"Could not locate '{slug}' dataset under /kaggle/input "
        f"(checked {direct} and /kaggle/input/datasets/*/{slug})"
    )

FEATURES_DIR = _resolve_dataset_dir("trace-the-ace-features")
OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_FOLDS = 5

print(f"Resolved FEATURES_DIR: {FEATURES_DIR}")


In [ ]:
# --- Load Notebook A outputs --------------------------------------------
train_features_enriched = pd.read_csv(FEATURES_DIR / "train_features_enriched.csv")
cv_folds = pd.read_csv(FEATURES_DIR / "cv_folds.csv")

print("train_features_enriched:", train_features_enriched.shape)
print("cv_folds:               ", cv_folds.shape)
train_features_enriched.head()


In [ ]:
# --- Merge fold assignment onto the response-level dataframe ----------------
df = train_features_enriched.merge(cv_folds, on="session_id", how="left")
assert df["fold"].isna().sum() == 0, "every response must have a non-null fold assignment"
df["fold"] = df["fold"].astype(int)

print(df["fold"].value_counts().sort_index())
print(f"base rate is_correct: {df['is_correct'].mean():.4f}")


## 2. Baseline Model

In [ ]:
# --- Explicit feature list (do not silently .filter columns) ----------------
FEATURE_COLS = [
    "feat_n_student_words",
    "feat_n_tutor_words",
    "feat_student_word_share",
    "feat_numeric_turns_per_word",
    "feat_digit_chars_per_word",
    "feat_number_word_count",
    "feat_hint_count",
    "feat_correction_count",
    "feat_question_count_tutor",
    "feat_question_count_student",
    "feat_turn_count",
    "feat_avg_turn_length_student",
    "feat_avg_turn_length_tutor",
    "feat_turn_gap_mean_s",
    "feat_turn_gap_std_s",
    "feat_late_session_student_share",
    "feat_embed_sim_objective_mean",
]
missing = set(FEATURE_COLS) - set(df.columns)
assert not missing, f"expected feature columns missing from dataframe: {missing}"
print(f"{len(FEATURE_COLS)} feature columns confirmed present")

X_all = df[FEATURE_COLS].values
y_all = df["is_correct"].values
fold_all = df["fold"].values


In [ ]:
# --- 5-fold out-of-fold training + prediction --------------------------------
# For each fold f: train on rows NOT in fold f, predict on rows IN fold f.
# By the end every response_id has exactly one OOF prediction -- required for
# unbiased stacking in Notebook D (Context.md sec 7: never fit on a session
# that appears in the validation fold for that split).

def make_pipeline():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ])

oof_pred = np.full(len(df), np.nan)
per_fold_metrics = []

for fold in range(N_FOLDS):
    train_mask = fold_all != fold
    val_mask = fold_all == fold

    X_train, y_train = X_all[train_mask], y_all[train_mask]
    X_val, y_val = X_all[val_mask], y_all[val_mask]

    pipe = make_pipeline()
    pipe.fit(X_train, y_train)
    val_pred = pipe.predict_proba(X_val)[:, 1]
    oof_pred[val_mask] = val_pred

    fold_log_loss = log_loss(y_val, val_pred)
    fold_auc = roc_auc_score(y_val, val_pred)

    # Base-rate floor: train-fold mean of is_correct, predicted constant for
    # every validation row in this fold (per Context.md sec 7 reporting standard).
    base_rate = y_train.mean()
    base_rate_pred = np.full(len(y_val), base_rate)
    base_rate_log_loss = log_loss(y_val, base_rate_pred, labels=[0, 1])
    base_rate_auc = roc_auc_score(y_val, base_rate_pred)

    per_fold_metrics.append({
        "fold": fold,
        "n_train": int(train_mask.sum()),
        "n_val": int(val_mask.sum()),
        "log_loss": fold_log_loss,
        "auc": fold_auc,
        "base_rate_log_loss": base_rate_log_loss,
        "base_rate_auc": base_rate_auc,
    })
    print(f"fold {fold}: log_loss={fold_log_loss:.4f} auc={fold_auc:.4f} "
          f"| base-rate floor log_loss={base_rate_log_loss:.4f} auc={base_rate_auc:.4f}")

assert np.isnan(oof_pred).sum() == 0, "every response must get exactly one OOF prediction"
df["pred_baseline"] = oof_pred


In [ ]:
# --- Aggregate metrics: mean/std across folds, and overall on stacked OOF ---
fold_log_losses = np.array([m["log_loss"] for m in per_fold_metrics])
fold_aucs = np.array([m["auc"] for m in per_fold_metrics])
fold_base_rate_log_losses = np.array([m["base_rate_log_loss"] for m in per_fold_metrics])
fold_base_rate_aucs = np.array([m["base_rate_auc"] for m in per_fold_metrics])

overall_log_loss = log_loss(df["is_correct"], df["pred_baseline"])
overall_auc = roc_auc_score(df["is_correct"], df["pred_baseline"])

print("=" * 60)
print(f"CV log loss: mean={fold_log_losses.mean():.4f} std={fold_log_losses.std():.4f}")
print(f"CV AUC:      mean={fold_aucs.mean():.4f} std={fold_aucs.std():.4f}")
print(f"Base-rate floor log loss: mean={fold_base_rate_log_losses.mean():.4f} "
      f"std={fold_base_rate_log_losses.std():.4f}")
print(f"Base-rate floor AUC:      mean={fold_base_rate_aucs.mean():.4f} "
      f"std={fold_base_rate_aucs.std():.4f}")
print("-" * 60)
print(f"Overall (stacked OOF) log loss: {overall_log_loss:.4f}")
print(f"Overall (stacked OOF) AUC:      {overall_auc:.4f}")
print("=" * 60)


In [ ]:
# --- Logistic regression coefficients (standardized), sorted by |magnitude| --
# Coefficients are on the StandardScaler-transformed features, so they're
# directly comparable across features regardless of each feature's raw scale.
# Fit one reference pipeline on ALL folds' combined training signal (the
# final-fit pipeline from section 3) purely to report coefficients here --
# the OOF predictions above already came from the proper per-fold models.
coef_pipeline = make_pipeline()
coef_pipeline.fit(X_all, y_all)
coefs = coef_pipeline.named_steps["clf"].coef_[0]

coef_df = (
    pd.DataFrame({"feature": FEATURE_COLS, "coef": coefs})
    .assign(abs_coef=lambda d: d["coef"].abs())
    .sort_values("abs_coef", ascending=False)
    .reset_index(drop=True)
)
print(coef_df[["feature", "coef"]].to_string(index=False))


**Reading the coefficients**: features with large positive coefficients push
predictions toward `is_correct=1` (correct answer); large negative coefficients
push toward incorrect. Whichever features surface at the top of `coef_df`
above are the model's strongest linear signals for correctness -- worth
highlighting in the write-up's "key findings" section, but keep in mind this
is a linear model on hand-built features, so it can't capture interactions or
nonlinear effects the LLM stage (Notebook C) might pick up.


## 3. Final Fit

Refit the same pipeline on ALL training data to produce the final baseline
model artifact for potential future non-stacked use. This is a *different*
model than any of the 5 per-fold models used to generate OOF predictions
above -- it has seen every training response, so its own predictions on
training data would be optimistically biased and must never be used as OOF
predictions for stacking.


In [ ]:
final_pipeline = make_pipeline()
final_pipeline.fit(X_all, y_all)
print("Final pipeline fit on all", len(df), "training responses.")


## 4. Outputs

**Important**: `oof_predictions_baseline.csv` holds the out-of-fold
predictions from section 2 (each response scored by a model that never saw
its session during training) -- NOT predictions from the section 3 final-fit
model, which has seen all training data and would leak information if used
for stacking in Notebook D.


In [ ]:
# --- oof_predictions_baseline.csv ---
oof_predictions = df[["response_id", "pred_baseline"]].copy()
oof_predictions.to_csv(OUT_DIR / "oof_predictions_baseline.csv", index=False)
print(oof_predictions.shape)
oof_predictions.head()


In [ ]:
# --- baseline_model.pkl (the section-3 full-data-fit pipeline) ---
joblib.dump(final_pipeline, OUT_DIR / "baseline_model.pkl")
print("saved baseline_model.pkl")


In [ ]:
# --- baseline_metrics.json ---
metrics = {
    "model": "logistic_regression_baseline",
    "feature_cols": FEATURE_COLS,
    "n_folds": N_FOLDS,
    "per_fold": per_fold_metrics,
    "cv_mean": {"log_loss": float(fold_log_losses.mean()), "auc": float(fold_aucs.mean())},
    "cv_std": {"log_loss": float(fold_log_losses.std()), "auc": float(fold_aucs.std())},
    "oof_overall": {"log_loss": float(overall_log_loss), "auc": float(overall_auc)},
    "base_rate_floor": {
        "mean_log_loss": float(fold_base_rate_log_losses.mean()),
        "mean_auc": float(fold_base_rate_aucs.mean()),
    },
    "coefficients": coef_df.to_dict(orient="records"),
}

with open(OUT_DIR / "baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(json.dumps({k: v for k, v in metrics.items() if k != "coefficients"}, indent=2))


In [ ]:
for fname in ["oof_predictions_baseline.csv", "baseline_model.pkl", "baseline_metrics.json"]:
    p = OUT_DIR / fname
    print(f"{fname}: {p.stat().st_size / 1e3:.1f} KB")
